# Advanced GINE + Frozen ESM-2 Drug–Target Affinity Prediction

This notebook evaluates the advanced drug–target affinity model on the **DAVIS** dataset.

The model extends the baseline experiments in three main ways:

- the drug encoder uses **GINE**, allowing chemical bond features to participate in message passing;
- protein sequences are represented using **frozen ESM-2 residue embeddings** followed by a task-specific 1D CNN;
- atom and residue representations interact through **bidirectional cross-attention** before pooling and affinity regression.

The advanced model is evaluated using the same three settings used for the DAVIS baselines:

- Random split
- Cold-drug split
- Cold-target split

DAVIS is used for the main advanced experiment so that the advanced architecture can be compared directly with the DAVIS GCN + CNN, GAT + CNN, and GIN + CNN baselines.

## Notebook workflow

```text
Drug SMILES                                      Protein AA Sequence
     │                                                   │
     ▼                                                   ▼
RDKit molecular graph                              Frozen ESM-2
atoms + bond features                              residue embeddings
     │                                                   │
     ▼                                                   ▼
4-layer GINE                                       projection to 128
Residual + LayerNorm                                    │
     │                                                   ▼
     │                                             3-layer 1D CNN
     │                                             kernels 7, 5, 3
     ▼                                                   ▼
Atom representations                              Residue representations
[B, N_atoms, 128]                                 [B, L, 128]
     │                                                   │
     └──────────────────────┬────────────────────────────┘
                            ▼
                Bidirectional Cross-Attention
                  atoms query residues
                  residues query atoms
                  MHA + residual + norm + FFN
                            │
              ┌─────────────┴─────────────┐
              ▼                           ▼
      Updated atom features       Updated residue features
              │                           │
              ▼                           ▼
      Attention pooling           Attention pooling
              │                           │
              ▼                           ▼
       Drug vector [B,128]       Protein vector [B,128]
              └─────────────┬─────────────┘
                            ▼
                      Concatenate
                        [B,256]
                            │
                            ▼
                    MLP regressor
                   256 → 512 → 256 → 1
                            │
                            ▼
                       Predicted pKd
                            │
          ┌─────────────────┼──────────────────┐
          ▼                 ▼                  ▼
       Random          Cold-drug          Cold-target
          │                 │                  │
          ▼                 ▼                  ▼
        Train / validate / test with early stopping
          │                 │                  │
          └─────────────────┼──────────────────┘
                            ▼
                  MSE, RMSE, MAE, CI, Pearson
                            │
                            ▼
                Save checkpoints, predictions,
                     figures and metrics
```

Frozen ESM-2 is used only as a feature extractor. Each unique DAVIS protein is embedded once and the residue representations are cached for reuse during training.

# Part A — Setup and DAVIS preparation

## 1. Imports

In [ ]:
import gc
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer

## 2. Locate the project folder

In [ ]:
def find_project_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/final_aml_gnn_dta"),
    ]

    content_dir = Path("/content")
    if content_dir.exists():
        candidates.extend(
            path.parent
            for path in content_dir.glob("*/config.py")
        )

    for candidate in candidates:
        if (
            (candidate / "config.py").exists()
            and (candidate / "src").exists()
        ):
            return candidate.resolve()

    raise FileNotFoundError(
        "Project root could not be found. "
        "Run this notebook from final_aml_gnn_dta or clone/copy "
        "the project into the current environment."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

## 3. Project settings and shared functions

The advanced model uses the same DAVIS data splits, random seed, maximum epochs, early-stopping behaviour, loss function, and evaluation metrics as the baseline experiments.

A smaller training batch size is used because cross-attention retains atom-level and residue-level representations.

In [ ]:
from config import (
    CHECKPOINTS_DIR,
    DAVIS_DIR,
    FIGURES_DIR,
    LEARNING_RATE,
    MAX_EPOCHS,
    MAX_PROTEIN_LENGTH,
    METRICS_DIR,
    PREDICTIONS_DIR,
    RANDOM_SEED,
    RESULTS_DIR,
    TRAIN_RATIO,
    VALIDATION_RATIO,
    TEST_RATIO,
)

from src.models import (
    AdvancedDTAModel,
    count_trainable_parameters,
)

from src.preprocessing import (
    ATOM_FEATURE_DIM,
    BOND_FEATURE_DIM,
    AdvancedInteractionDataset,
    advanced_collate_fn,
    build_drug_graph_cache,
    build_esm_embedding_cache,
    check_cold_split,
    check_esm_embedding_cache,
    cold_drug_split,
    cold_target_split,
    load_deepdta_dataset,
    load_esm_embedding_cache,
    random_split,
    save_esm_embedding_cache,
)

from src.training import (
    evaluate_model,
    fit_model,
)

from src.utils import (
    get_device,
    save_predictions_csv,
    set_seed,
)

set_seed(RANDOM_SEED)

device = get_device()

ESM_MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

ADVANCED_BATCH_SIZE = 16
ESM_CACHE_BATCH_SIZE = 8

ESM_CACHE_DIR = RESULTS_DIR / "esm_embeddings"
ESM_CACHE_DIR.mkdir(parents=True, exist_ok=True)

ESM_CACHE_PATH = (
    ESM_CACHE_DIR
    / "davis_esm2_t6_residue_embeddings.pt"
)

print("Training device:", device)
print("Random seed:", RANDOM_SEED)
print("Advanced batch size:", ADVANCED_BATCH_SIZE)
print("Maximum epochs:", MAX_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("ESM-2 model:", ESM_MODEL_NAME)

## 4. Load the DAVIS dataset

The DAVIS dissociation constants are converted from Kd in nM to **pKd** by the shared preprocessing function. Higher pKd therefore represents stronger binding.

In [ ]:
davis_df = load_deepdta_dataset(
    dataset_dir=DAVIS_DIR,
    dataset_name="davis",
    transform_davis=True,
)

print("Interactions:", len(davis_df))
print("Unique drugs:", davis_df["drug_id"].nunique())
print("Unique targets:", davis_df["target_id"].nunique())
print(
    "pKd range:",
    f'{davis_df["affinity"].min():.3f}',
    "to",
    f'{davis_df["affinity"].max():.3f}',
)

display(davis_df.head())

## 5. Prepare and verify molecular graphs

The advanced GINE encoder requires both atom features and bond features.  
Every graph is therefore checked to confirm that `edge_attr` is present and has the expected bond-feature dimension.

In [ ]:
drug_graph_cache = build_drug_graph_cache(
    davis_df
)

invalid_graphs = []

for drug_id, graph in drug_graph_cache.items():
    if (
        graph.edge_attr is None
        or graph.edge_attr.ndim != 2
        or graph.edge_attr.shape[1] != BOND_FEATURE_DIM
    ):
        invalid_graphs.append(drug_id)

print("Cached drug graphs:", len(drug_graph_cache))
print("Atom feature dimension:", ATOM_FEATURE_DIM)
print("Bond feature dimension:", BOND_FEATURE_DIM)
print("Invalid bond-feature graphs:", len(invalid_graphs))

assert not invalid_graphs, (
    "Some drug graphs do not contain valid bond features."
)

example_drug_id = next(iter(drug_graph_cache))
example_graph = drug_graph_cache[example_drug_id]

print()
print("Example drug:", example_drug_id)
print("Node features:", tuple(example_graph.x.shape))
print("Edge index:", tuple(example_graph.edge_index.shape))
print("Bond features:", tuple(example_graph.edge_attr.shape))

# Part B — Frozen ESM-2 residue embeddings

ESM-2 is not trained in this project. It is used as a **frozen protein feature extractor**.

The cache is created only when it does not already exist. Once created, subsequent runs load the stored residue embeddings directly.

## 6. Create or load the ESM-2 cache

For portability, ESM-2 embedding generation uses CUDA when available and otherwise uses CPU.  
The advanced DTA model itself still uses the best device returned by the project utility.

In [ ]:
if ESM_CACHE_PATH.exists():
    print("Loading existing ESM-2 cache...")
    protein_embedding_cache = load_esm_embedding_cache(
        ESM_CACHE_PATH
    )

else:
    print("Creating ESM-2 residue embeddings...")

    esm_device = (
        torch.device("cuda")
        if torch.cuda.is_available()
        else torch.device("cpu")
    )

    print("ESM-2 device:", esm_device)

    tokenizer = AutoTokenizer.from_pretrained(
        ESM_MODEL_NAME
    )

    esm_model = AutoModel.from_pretrained(
        ESM_MODEL_NAME
    )

    protein_embedding_cache = build_esm_embedding_cache(
        dataframe=davis_df,
        tokenizer=tokenizer,
        esm_model=esm_model,
        device=esm_device,
        max_length=MAX_PROTEIN_LENGTH,
        batch_size=ESM_CACHE_BATCH_SIZE,
        storage_dtype=torch.float16,
    )

    save_esm_embedding_cache(
        protein_embedding_cache,
        ESM_CACHE_PATH,
    )

    del esm_model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("ESM-2 cache saved.")

print("Cache path:", ESM_CACHE_PATH)

## 7. Validate the ESM-2 cache

In [ ]:
cache_is_valid = check_esm_embedding_cache(
    davis_df,
    protein_embedding_cache,
)

print("Cache valid:", cache_is_valid)
print(
    "Cached target embeddings:",
    len(protein_embedding_cache),
)

assert cache_is_valid, (
    "The ESM-2 cache is incomplete or invalid."
)

sample_target_id = next(
    iter(protein_embedding_cache)
)

sample_embedding = protein_embedding_cache[
    sample_target_id
]

ESM_DIM = sample_embedding.shape[1]

print()
print("Example target:", sample_target_id)
print(
    "Residue embedding shape:",
    tuple(sample_embedding.shape),
)
print("Stored dtype:", sample_embedding.dtype)
print("ESM embedding dimension:", ESM_DIM)

# Part C — Advanced model sanity check

Before running the full experiments, a small batch is passed through the advanced model.  
This verifies that the GINE branch, cached ESM-2 embeddings, padding masks, cross-attention, pooling, and regression head are shape-compatible.

## 8. Create a small advanced-model batch

In [ ]:
sanity_df = davis_df.head(8).copy()

sanity_dataset = AdvancedInteractionDataset(
    dataframe=sanity_df,
    drug_graph_cache=drug_graph_cache,
    protein_embedding_cache=protein_embedding_cache,
)

sanity_loader = DataLoader(
    sanity_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=advanced_collate_fn,
)

sanity_batch = next(iter(sanity_loader))

print(
    "Protein embeddings:",
    tuple(sanity_batch["protein_embeddings"].shape),
)
print(
    "Protein mask:",
    tuple(sanity_batch["protein_mask"].shape),
)
print(
    "Affinity labels:",
    tuple(sanity_batch["y"].shape),
)
print(
    "Batched graph nodes:",
    sanity_batch["graph"].num_nodes,
)

## 9. Build the advanced model and test one forward pass

In [ ]:
sanity_model = AdvancedDTAModel(
    esm_dim=ESM_DIM,
    hidden_dim=128,
    attention_heads=4,
    dropout=0.20,
    regressor_dropout=0.30,
).to(device)

print(
    "Trainable parameters:",
    f"{count_trainable_parameters(sanity_model):,}",
)

with torch.no_grad():
    sanity_predictions = sanity_model(
        sanity_batch["graph"].to(device),
        sanity_batch["protein_embeddings"].to(device),
        sanity_batch["protein_mask"].to(device),
    )

print(
    "Prediction shape:",
    tuple(sanity_predictions.shape),
)
print(
    "Expected shape:",
    tuple(sanity_batch["y"].shape),
)

assert (
    sanity_predictions.shape
    == sanity_batch["y"].shape
)

del sanity_model
del sanity_batch
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    torch.mps.empty_cache()

print("Advanced-model forward-pass check passed.")

## 10. Helper functions for the three advanced experiments

In [ ]:
def create_advanced_loaders(
    train_df,
    validation_df,
    test_df,
):
    train_dataset = AdvancedInteractionDataset(
        train_df,
        drug_graph_cache,
        protein_embedding_cache,
    )

    validation_dataset = AdvancedInteractionDataset(
        validation_df,
        drug_graph_cache,
        protein_embedding_cache,
    )

    test_dataset = AdvancedInteractionDataset(
        test_df,
        drug_graph_cache,
        protein_embedding_cache,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=ADVANCED_BATCH_SIZE,
        shuffle=True,
        collate_fn=advanced_collate_fn,
        num_workers=0,
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=ADVANCED_BATCH_SIZE,
        shuffle=False,
        collate_fn=advanced_collate_fn,
        num_workers=0,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=ADVANCED_BATCH_SIZE,
        shuffle=False,
        collate_fn=advanced_collate_fn,
        num_workers=0,
    )

    return (
        train_loader,
        validation_loader,
        test_loader,
    )


def print_split_summary(
    train_df,
    validation_df,
    test_df,
):
    summary = pd.DataFrame(
        {
            "partition": [
                "Train",
                "Validation",
                "Test",
            ],
            "interactions": [
                len(train_df),
                len(validation_df),
                len(test_df),
            ],
            "unique_drugs": [
                train_df["drug_id"].nunique(),
                validation_df["drug_id"].nunique(),
                test_df["drug_id"].nunique(),
            ],
            "unique_targets": [
                train_df["target_id"].nunique(),
                validation_df["target_id"].nunique(),
                test_df["target_id"].nunique(),
            ],
        }
    )

    display(summary)


def plot_advanced_history(
    history,
    split_name,
):
    history_df = pd.DataFrame(history)

    plt.figure(figsize=(8, 5))
    plt.plot(
        history_df["epoch"],
        history_df["train_loss"],
        label="Training MSE",
    )
    plt.plot(
        history_df["epoch"],
        history_df["validation_loss"],
        label="Validation MSE",
    )
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.title(
        "DAVIS Advanced GINE + ESM-2 — "
        f"{split_name}"
    )
    plt.legend()
    plt.tight_layout()

    figure_path = (
        FIGURES_DIR
        / (
            "davis_advanced_gine_esm2_"
            f"{split_name}_training.png"
        )
    )

    plt.savefig(
        figure_path,
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()

    return history_df


def run_advanced_experiment(
    split_name,
    train_df,
    validation_df,
    test_df,
):
    print(f"Running split: {split_name}")
    print("-" * 60)

    (
        train_loader,
        validation_loader,
        test_loader,
    ) = create_advanced_loaders(
        train_df,
        validation_df,
        test_df,
    )

    model = AdvancedDTAModel(
        esm_dim=ESM_DIM,
        hidden_dim=128,
        attention_heads=4,
        dropout=0.20,
        regressor_dropout=0.30,
    )

    trainable_parameters = (
        count_trainable_parameters(model)
    )

    print(
        "Trainable parameters:",
        f"{trainable_parameters:,}",
    )

    checkpoint_path = (
        CHECKPOINTS_DIR
        / (
            "davis_advanced_gine_esm2_"
            f"{split_name}.pt"
        )
    )

    training_result = fit_model(
        model=model,
        train_loader=train_loader,
        validation_loader=validation_loader,
        device=device,
        checkpoint_path=checkpoint_path,
        max_epochs=MAX_EPOCHS,
        learning_rate=LEARNING_RATE,
    )

    history_df = plot_advanced_history(
        training_result["history"],
        split_name,
    )

    test_result = evaluate_model(
        model=model,
        data_loader=test_loader,
        device=device,
    )

    metrics = test_result["metrics"]

    predictions_df = test_df.copy()

    predictions_df["true_affinity"] = (
        test_result["y_true"]
    )
    predictions_df["predicted_affinity"] = (
        test_result["y_pred"]
    )
    predictions_df["prediction_error"] = (
        predictions_df["predicted_affinity"]
        - predictions_df["true_affinity"]
    )

    prediction_path = (
        PREDICTIONS_DIR
        / (
            "davis_advanced_gine_esm2_"
            f"{split_name}.csv"
        )
    )

    save_predictions_csv(
        predictions_df,
        prediction_path,
    )

    result_row = {
        "dataset": "DAVIS",
        "model": (
            "GINE + frozen ESM-2 + CNN "
            "+ cross-attention"
        ),
        "split": split_name,
        "best_epoch": training_result["best_epoch"],
        "completed_epochs": training_result[
            "completed_epochs"
        ],
        "best_validation_mse": training_result[
            "best_validation_mse"
        ],
        "test_mse": metrics["mse"],
        "test_rmse": metrics["rmse"],
        "test_mae": metrics["mae"],
        "ci": metrics["ci"],
        "pearson": metrics["pearson"],
        "trainable_parameters": trainable_parameters,
    }

    print()
    print("Best epoch:", result_row["best_epoch"])
    print(
        "Best validation MSE:",
        f'{result_row["best_validation_mse"]:.4f}',
    )
    print(
        "Test RMSE:",
        f'{result_row["test_rmse"]:.4f}',
    )
    print(
        "Test MAE:",
        f'{result_row["test_mae"]:.4f}',
    )
    print(
        "Concordance index:",
        f'{result_row["ci"]:.4f}',
    )
    print(
        "Pearson correlation:",
        f'{result_row["pearson"]:.4f}',
    )

    # Move the completed model back to CPU before starting
    # the next split so GPU memory is released.
    model = model.to("cpu")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif (
        hasattr(torch.backends, "mps")
        and torch.backends.mps.is_available()
    ):
        torch.mps.empty_cache()

    return {
        "model": model,
        "result": result_row,
        "history": history_df,
        "predictions": predictions_df,
        "checkpoint_path": checkpoint_path,
        "prediction_path": prediction_path,
    }

# Part D — Random split experiment

The random split uses the same seed and split ratios as the DAVIS baseline notebooks.  
Drug and target identities may therefore appear in more than one partition, while the individual interaction pairs are separated.

## 11. Create the random split

In [ ]:
random_train_df, random_validation_df, random_test_df = random_split(
    davis_df,
    train_ratio=TRAIN_RATIO,
    validation_ratio=VALIDATION_RATIO,
    test_ratio=TEST_RATIO,
    seed=RANDOM_SEED,
)

print_split_summary(
    random_train_df,
    random_validation_df,
    random_test_df,
)

print(
    "Train-test drug overlap:",
    len(
        set(random_train_df["drug_id"])
        & set(random_test_df["drug_id"])
    ),
)

print(
    "Train-test target overlap:",
    len(
        set(random_train_df["target_id"])
        & set(random_test_df["target_id"])
    ),
)

## 12. Train and evaluate the advanced model on the random split

In [ ]:
random_result = run_advanced_experiment(
    split_name="random",
    train_df=random_train_df,
    validation_df=random_validation_df,
    test_df=random_test_df,
)

## 13. Inspect random-split predictions

In [ ]:
display(
    random_result["predictions"][
        [
            "drug_id",
            "target_id",
            "true_affinity",
            "predicted_affinity",
            "prediction_error",
        ]
    ].head(10)
)

# Part E — Cold-drug experiment

The cold-drug split contains no shared drugs between training, validation, and test partitions.

This experiment evaluates whether the advanced molecular encoder can generalise to compounds that were not observed during training.

## 14. Create and verify the cold-drug split

In [ ]:
(
    cold_drug_train_df,
    cold_drug_validation_df,
    cold_drug_test_df,
) = cold_drug_split(
    davis_df,
    train_ratio=TRAIN_RATIO,
    validation_ratio=VALIDATION_RATIO,
    test_ratio=TEST_RATIO,
    seed=RANDOM_SEED,
)

print_split_summary(
    cold_drug_train_df,
    cold_drug_validation_df,
    cold_drug_test_df,
)

cold_drug_is_valid = check_cold_split(
    cold_drug_train_df,
    cold_drug_validation_df,
    cold_drug_test_df,
    column="drug_id",
)

print("No drug overlap:", cold_drug_is_valid)

assert cold_drug_is_valid, (
    "Cold-drug split contains overlapping drugs."
)

## 15. Train and evaluate the advanced model on the cold-drug split

In [ ]:
cold_drug_result = run_advanced_experiment(
    split_name="cold_drug",
    train_df=cold_drug_train_df,
    validation_df=cold_drug_validation_df,
    test_df=cold_drug_test_df,
)

## 16. Inspect cold-drug predictions

In [ ]:
display(
    cold_drug_result["predictions"][
        [
            "drug_id",
            "target_id",
            "true_affinity",
            "predicted_affinity",
            "prediction_error",
        ]
    ].head(10)
)

# Part F — Cold-target experiment

The cold-target split contains no shared protein targets between training, validation, and test partitions.

This is especially relevant to the advanced model because the protein branch uses pretrained frozen ESM-2 residue representations before task-specific CNN and cross-attention processing.

## 17. Create and verify the cold-target split

In [ ]:
(
    cold_target_train_df,
    cold_target_validation_df,
    cold_target_test_df,
) = cold_target_split(
    davis_df,
    train_ratio=TRAIN_RATIO,
    validation_ratio=VALIDATION_RATIO,
    test_ratio=TEST_RATIO,
    seed=RANDOM_SEED,
)

print_split_summary(
    cold_target_train_df,
    cold_target_validation_df,
    cold_target_test_df,
)

cold_target_is_valid = check_cold_split(
    cold_target_train_df,
    cold_target_validation_df,
    cold_target_test_df,
    column="target_id",
)

print(
    "No target overlap:",
    cold_target_is_valid,
)

assert cold_target_is_valid, (
    "Cold-target split contains overlapping targets."
)

## 18. Train and evaluate the advanced model on the cold-target split

In [ ]:
cold_target_result = run_advanced_experiment(
    split_name="cold_target",
    train_df=cold_target_train_df,
    validation_df=cold_target_validation_df,
    test_df=cold_target_test_df,
)

## 19. Inspect cold-target predictions

In [ ]:
display(
    cold_target_result["predictions"][
        [
            "drug_id",
            "target_id",
            "true_affinity",
            "predicted_affinity",
            "prediction_error",
        ]
    ].head(10)
)

# Part G — Advanced-model results summary

The three advanced experiments are combined into one metrics file for later comparison with the three DAVIS baseline models.

## 20. Combine and save advanced-model metrics

In [ ]:
advanced_results_df = pd.DataFrame(
    [
        random_result["result"],
        cold_drug_result["result"],
        cold_target_result["result"],
    ]
)

metrics_path = (
    METRICS_DIR
    / "davis_advanced_gine_esm2.csv"
)

advanced_results_df.to_csv(
    metrics_path,
    index=False,
)

display(
    advanced_results_df.round(4)
)

print("Metrics saved to:", metrics_path)

## 21. Compare test RMSE across the three splits

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    advanced_results_df["split"],
    advanced_results_df["test_rmse"],
)

plt.xlabel("Data split")
plt.ylabel("Test RMSE")
plt.title(
    "DAVIS Advanced GINE + ESM-2 — RMSE by split"
)
plt.tight_layout()

comparison_figure_path = (
    FIGURES_DIR
    / "davis_advanced_gine_esm2_split_comparison.png"
)

plt.savefig(
    comparison_figure_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

print(
    "Comparison figure saved to:",
    comparison_figure_path,
)

## 22. Final advanced-model summary

In [ ]:
summary_columns = [
    "split",
    "best_epoch",
    "best_validation_mse",
    "test_rmse",
    "test_mae",
    "ci",
    "pearson",
]

final_summary = (
    advanced_results_df[
        summary_columns
    ]
    .sort_values("test_rmse")
    .reset_index(drop=True)
)

display(
    final_summary.round(4)
)

best_split = final_summary.iloc[0]

print(
    "Lowest advanced-model test RMSE:",
    best_split["split"],
    f'({best_split["test_rmse"]:.4f})',
)

print()
print("Saved outputs")
print("-" * 55)
print("Metrics:", metrics_path)
print(
    "Random predictions:",
    random_result["prediction_path"],
)
print(
    "Cold-drug predictions:",
    cold_drug_result["prediction_path"],
)
print(
    "Cold-target predictions:",
    cold_target_result["prediction_path"],
)
print("ESM-2 cache:", ESM_CACHE_PATH)
print("Figures folder:", FIGURES_DIR)
print("Checkpoints folder:", CHECKPOINTS_DIR)

### Notebook completion

This notebook completes the advanced DAVIS experiment using:

**4-layer GINE + frozen ESM-2 + 3-layer CNN + bidirectional cross-attention + attention pooling + MLP regression.**

The saved `davis_advanced_gine_esm2.csv` file can later be compared directly with:

```text
davis_gcn_cnn.csv
davis_gat_cnn.csv
davis_gin_cnn.csv
```

The AML candidate-ranking stage is intentionally kept separate and will use the trained model later in the AML screening notebook.